In [1]:
from catalyst.debug.compiler_functions import get_compilation_stage
import pennylane as qml
from catalyst.third_party.oqd import OQDDevice

import os
import shutil
import pathlib
import numpy as np

from functools import partial

########################################################################################

for f in os.listdir():
    if f.startswith("oqd_circuit") and os.path.isdir(f):
        shutil.rmtree(pathlib.Path(f))

compile_results = pathlib.Path("oqd_circuit_mipt")
openapl_file_name = "oqd_circuit_mipt.openapl.json"


toml_files = {
    "device-toml-loc": "/home/user/oqd-catalyst/scripts/calibration_data/device.toml",
    "qubit-toml-loc": "/home/user/oqd-catalyst/scripts/calibration_data/qubit.toml",
    "gate-to-pulse-toml-loc": "/home/user/oqd-catalyst/scripts/calibration_data/gate.toml",
}

toml_files = " ".join([f"{k}={v}" for k, v in toml_files.items()])


OQD_PIPELINES = [
    (
        "DeviceAgnosticPipeline",
        [
            "quantum-compilation-stage",
            "hlo-lowering-stage",
            "gradient-lowering-stage",
            "bufferization-stage",
        ],
    ),
    (
        "IonDecompositionStage",
        [
            "func.func(ions-decomposition)",
            "func.func(merge-rotations)",
            # "func.func(prune-zero-rotations)",
        ],
    ),
    (
        "IonDialectLoweringStage",
        [
            f"func.func(gates-to-pulses{{{toml_files}}})",
        ],
    ),
    ("IonToLLVMDialectConversion", ["convert-ion-to-llvm"]),
    ("MLIRToLLVMDialectConversion", ["llvm-dialect-lowering-stage"]),
]


N = 3
T = 2 * N

# Measurement rate.  Vary this to cross the MIPT phase transition.
p = 0.5

# Pre-generate a fixed (reproducible) measurement pattern: shape (T, N), dtype bool.
np.random.seed(3141592653)
measurement_record = np.random.random((T, N)) < p

print(f"No. of measurements: \033[1;32m{measurement_record.sum()}\033[0m")

measurement_record = measurement_record.tolist()


oqd_dev = OQDDevice(
    backend="default",
    wires=N,
    openapl_file_name=(compile_results / openapl_file_name).as_posix(),
)


def block(num_qubits, block_rotation_record):
    for i in range(num_qubits // 2):
        qml.IsingXX(phi=np.pi / 2, wires=[2 * i, 2 * i + 1])

    for i in range(num_qubits):
        qml.Rot(block_rotation_record[i], np.pi / 2, -block_rotation_record[i], wires=i)

    for i in range(num_qubits // 2):
        qml.IsingXX(phi=np.pi / 2, wires=[2 * i + 1, (2 * i + 2) % num_qubits])

    for i in range(num_qubits):
        qml.Rot(block_rotation_record[i], np.pi / 2, -block_rotation_record[i], wires=i)


def measure_layer(num_qubits, meas_record_t):
    """Mid-circuit measurement layer: measure qubit i if meas_record_t[i] is True."""
    for i in range(num_qubits):
        if meas_record_t[i]:
            qml.measure(i)


rotation_record = np.random.randint(0, 3, (T, N)) * np.pi / 4


# @partial(
#     qml.transforms.decompose,
#     gate_set={qml.RX, qml.RY, qml.CNOT},
# )
@qml.set_shots(10)
@qml.qnode(oqd_dev)
def oqd_circuit_mipt():
    for i, rr in enumerate(rotation_record):
        block(N, rr)
        measure_layer(N, measurement_record[i])
    return qml.counts(wires=0)


QJIT_CIRCUIT = qml.qjit(
    oqd_circuit_mipt, pipelines=OQD_PIPELINES, keep_intermediate=True, verbose=True
)

print("{:=^100}".format("\033[1;32m Compiled circuit \033[0m"))
print(get_compilation_stage(QJIT_CIRCUIT, stage="IonDialectLoweringStage"))


No. of measurements: 8
[LIB] Running compiler driver in /home/user/oqd-catalyst/scripts/oqd_circuit_mipt
[SYSTEM] /home/user/oqd-catalyst/frontend/catalyst/utils/../../../mlir/build/bin/catalyst -o /home/user/oqd-catalyst/scripts/oqd_circuit_mipt/oqd_circuit_mipt.ll --module-name oqd_circuit_mipt --workspace /home/user/oqd-catalyst/scripts/oqd_circuit_mipt -verify-each=false --catalyst-pipeline DeviceAgnosticPipeline(quantum-compilation-stage;hlo-lowering-stage;gradient-lowering-stage;bufferization-stage),IonDecompositionStage(func.func(ions-decomposition);func.func(merge-rotations)),IonDialectLoweringStage(func.func(gates-to-pulses{device-toml-loc=/home/user/oqd-catalyst/scripts/calibration_data/device.toml qubit-toml-loc=/home/user/oqd-catalyst/scripts/calibration_data/qubit.toml gate-to-pulse-toml-loc=/home/user/oqd-catalyst/scripts/calibration_data/gate.toml})),IonToLLVMDialectConversion(convert-ion-to-llvm),MLIRToLLVMDialectConversion(llvm-dialect-lowering-stage), --keep-intermedi

In [2]:
import json

print(qml.draw(oqd_circuit_mipt)())

with open("oqd_circuit_mipt.draw.txt", "w") as f:
    f.write(qml.draw(oqd_circuit_mipt)())


QJIT_CIRCUIT()

print(json.dumps(json.load(open(compile_results / openapl_file_name)), indent=2))

0: ─╭IsingXX(1.57)─────────Rot(1.57,1.57,-1.57)──Rot(1.57,1.57,-1.57)────────────────────── ···
1: ─╰IsingXX(1.57)─────────Rot(1.57,1.57,-1.57)─╭IsingXX(1.57)─────────Rot(1.57,1.57,-1.57) ···
2: ──Rot(0.00,1.57,-0.00)───────────────────────╰IsingXX(1.57)─────────Rot(0.00,1.57,-0.00) ···

0: ··· ───────────────────────╭IsingXX(1.57)──Rot(1.57,1.57,-1.57)──Rot(1.57,1.57,-1.57) ···
1: ··· ──┤↗├──────────────────╰IsingXX(1.57)──Rot(0.00,1.57,-0.00)─╭IsingXX(1.57)─────── ···
2: ··· ──Rot(0.79,1.57,-0.79)──────────────────────────────────────╰IsingXX(1.57)─────── ···

0: ··· ─────────────────────────────────────────────╭IsingXX(1.57)──Rot(0.79,1.57,-0.79) ···
1: ··· ──Rot(0.00,1.57,-0.00)──┤↗├──────────────────╰IsingXX(1.57)──Rot(0.00,1.57,-0.00) ···
2: ··· ──Rot(0.79,1.57,-0.79)──Rot(0.79,1.57,-0.79)───────────────────────────────────── ···

0: ··· ──Rot(0.79,1.57,-0.79)──┤↗├──────────────────╭IsingXX(1.57)─────────Rot(0.79,1.57,-0.79) ···
1: ··· ─╭IsingXX(1.57)─────────Rot(0.00,1.57,-0.00)

In [3]:
from oqd_core.interface.atomic import AtomicCircuit
from oqd_core.compiler.atomic.canonicalize import canonicalize_atomic_circuit_factory
from oqd_compiler_infrastructure import Chain, Post
from oqd_bare_metal.compiler.codegen import AtomicToTestbenchV2
from oqd_bare_metal.compiler.optim import (
    SpectrumCoreRemapping,
    SpectrumPrune,
    SpectrumUnwrapResets,
)
import ast_comments as ast


circuit = AtomicCircuit.model_validate_json(
    json.dumps(json.load(open(compile_results / openapl_file_name)), indent=2)
)


compiler = Chain(
    canonicalize_atomic_circuit_factory(),
    Post(
        AtomicToTestbenchV2(
            device_params="./calibration_data/testbench_params.toml",
        ),
    ),
)
optimization_pass = Chain(
    Post(SpectrumCoreRemapping(device="raman_awg")),
    Post(SpectrumUnwrapResets()),
    Post(SpectrumPrune()),
)

unopt_artiq_experiment = compiler(circuit)
artiq_experiment = optimization_pass(unopt_artiq_experiment)

print(ast.unparse(ast.fix_missing_locations(artiq_experiment)))

with open(compile_results / "oqd_circuit_mipt.artiq.py", "w") as f:
    f.write(ast.unparse(ast.fix_missing_locations(artiq_experiment)))

import numpy as np
from artiq.experiment import *

class TestbenchV2Experiment(EnvExperiment):

    @rpc(flags={'async'})
    def transfer_data(self, key, index, value):
        self.mutate_dataset(key=key, index=index, value=value)

    def build(self):
        self.setattr_device('core')
        self.setattr_device('awg')
        self.setattr_device('ttl0')
        self.setattr_device('ttl4')
        self.setattr_device('ttl6')
        self.setattr_device('ttl7')
        self.setattr_device('urukul0_ch0')
        self.setattr_device('urukul0_ch1')
        self.setattr_device('urukul0_ch2')
        self.setattr_device('urukul0_ch3')
        self.setattr_device('urukul1_ch0')
        self.setattr_device('urukul1_ch1')
        self.setattr_device('urukul1_ch2')
        self.setattr_device('urukul1_ch3')

    def program_awg(self):
        # init Spectrum AWG
        self.awg.start()
        self.awg.card_mode('dds')
        self.awg.channel_enable_out(True)
        self.awg.channels_out

In [4]:
# from oqd_core.interface.atomic import AtomicCircuit
# from oqd_core.compiler.atomic.canonicalize import canonicalize_atomic_circuit_factory
# from oqd_compiler_infrastructure import Chain, Post
# from oqd_bare_metal.compiler.codegen import AtomicToBloodstoneV1
# from oqd_bare_metal.compiler.optim import (
#     SpectrumCoreRemapping,
#     SpectrumPrune,
#     SpectrumUnwrapResets,
# )
# import ast_comments as ast


# circuit = AtomicCircuit.model_validate_json(
#     json.dumps(json.load(open(compile_results / openapl_file_name)), indent=2)
# )


# compiler = Chain(
#     canonicalize_atomic_circuit_factory(),
#     Post(
#         AtomicToBloodstoneV1(device_params="./calibration_data/bloodstone_params.toml")
#     ),
# )
# optimization_pass = Chain(
#     Post(SpectrumCoreRemapping(device="raman_awg")),
#     Post(SpectrumUnwrapResets()),
#     Post(SpectrumPrune()),
# )

# unopt_artiq_experiment = compiler(circuit)
# artiq_experiment = optimization_pass(unopt_artiq_experiment)


# with open(compile_results / "oqd_circuit.artiq.py", "w") as f:
#     f.write(ast.unparse(ast.fix_missing_locations(artiq_experiment)))